# Workshop 2 — Units and celestial coordinates

## Representing astronomical quantities and positions correctly

Astronomical analysis involves numbers with physical meaning: distances, velocities, masses, angles and positions on the sky. A bare number such as `220` is ambiguous - does it mean metres per second, kilometres per second, parsecs, degrees, or something else?

In this workshop you will learn how Astropy helps us attach physical meaning to numbers and use that information to avoid errors in analysis.

You will learn how to:

- create and manipulate `Quantity` objects
- convert between compatible units
- use physical constants
- work with angular units
- create `SkyCoord` objects
- convert between coordinate frames
- calculate angular separations
- work with arrays of positions
- match sources between two astronomical catalogues

<figure style="text-align: center;">
    <img
      src="images/Celestial_Sphere.png"
      alt="Model of the equatorial coordinate system"
      width="300"
    >
    <figcaption>
      Figure 1: Model of the equatorial coordinate system. Declination (vertical arcs, degrees) and hour angle (horizontal arcs, hours) are shown. For hour angle, right ascension (horizontal arcs, degrees) can be used as an alternative (<a href="https://commons.wikimedia.org/wiki/File:Celestial_Sphere_-_Equatorial_Coordinate_System.png">ChristianReady</a>, <a href="https://creativecommons.org/licenses/by-sa/4.0">CC BY-SA 4.0</a>)
    </figcaption>
</figure>

## 1. Import the packages

The two key Astropy modules are:

- `astropy.units`, conventionally imported as `u`
- `astropy.coordinates`, from which we will use `SkyCoord`.

We will also use NumPy and a few physical constants.

In [1]:
import numpy as np

from astropy import units as u
from astropy import constants as const
from astropy.coordinates import SkyCoord

# Part A — Physical quantities and units

## 2. A number with a unit

Multiplying a number by an Astropy unit creates a `Quantity`.

In [2]:
distance = 8.2 * u.kpc
speed = 230 * u.km / u.s

print(distance)
print(speed)
print(type(distance))

8.2 kpc
230.0 km / s
<class 'astropy.units.quantity.Quantity'>


A `Quantity` contains both a numerical value and a unit.

In [3]:
print(distance.value)
print(distance.unit)

8.2
kpc


Arrays can have units too.

In [4]:
wavelengths = np.array([450, 850, 1300]) * u.um
wavelengths

<Quantity [ 450.,  850., 1300.] um>

## 3. Unit conversion

Use `.to()` to convert a quantity to a compatible unit.

In [5]:
print(distance.to(u.pc))
print(distance.to(u.lyr))
print(speed.to(u.m / u.s))

8200.0 pc
26744.822972772952 lyr
230000.0 m / s


Astropy checks dimensional consistency. For example, a distance cannot be converted directly into a velocity.

Try uncommenting the line below and inspect the error message.

In [6]:
# distance.to(u.km / u.s)

This is useful. Unit errors that might otherwise remain hidden in a calculation can become explicit exceptions.

### Exercise 1

Create a wavelength of 1.3 mm and convert it to:

- micrometres
- metres.

In [7]:
# Your code here

In [8]:
# Solution
lam = 1.3 * u.mm

print(lam.to(u.um))
print(lam.to(u.m))

1300.0000000000002 um
0.0013000000000000002 m


## 4. Units propagate through calculations

Astropy carries units through arithmetic.

In [9]:
time = 12 * u.yr
velocity = 15 * u.km / u.s

distance_travelled = velocity * time

print(distance_travelled)
print(distance_travelled.to(u.au))

180.0 km yr / s
37.97091478254577 AU


## 5. An astronomy example: an enclosed Galactic mass

Consider a simple circular-orbit estimate. The enclosed mass $M$ depends on the rotation speed $v$, the radius $R$, and Newton's gravitational constant $G$ via:

$$
M = \frac{v^2 R}{G}.
$$

We will use illustrative values for the Sun's Galactocentric radius and orbital speed...

In [10]:
R = 8.2 * u.kpc
v = 230 * u.km / u.s

M_enclosed = v**2 * R / const.G

print(M_enclosed)

6499258349190177.0 km2 kg kpc / m3


The result is expressed in a valid but inconvenient combination of units inherited from the quantities in the calculation. The `.decompose()` method reduces a quantity to irreducible base units. With Astropy's default unit system, the mass is therefore expressed in kilograms.

In [11]:
M_enclosed_kg = M_enclosed.decompose()
M_enclosed_kg

<Quantity 2.00546158e+41 kg>

Once the dimensions are clear, convert the mass in kilograms to the astronomically useful unit of solar masses with `.to()`.

In [12]:
M_enclosed_solar = M_enclosed_kg.to(u.Msun)
M_enclosed_solar

<Quantity 1.00857555e+11 solMass>

Without a unit-aware library, this calculation would require us to track kilometres, seconds, kiloparsecs and the units of `G` manually. Here Astropy first confirms that the result has dimensions of mass, then converts it into the unit most useful for the scientific context.

### Exercise 2

The escape speed from a point mass is

$$
v_{esc} = \sqrt{\frac{2GM}{R}}.
$$

Calculate the escape speed at a radius of 1 au from a star of mass 1 solar mass and express the result in km/s.

In [13]:
# Your code here

In [14]:
# Solution
M_star = 1.0 * u.Msun
R_orbit = 1.0 * u.au

v_escape = np.sqrt(2 * const.G * M_star / R_orbit)

print(v_escape.to(u.km / u.s))

42.12191513663223 km / s


# Part B — Angles

## 6. Angles are quantities

Degrees, arcminutes and arcseconds are ordinary Astropy units.

In [15]:
angle = 1.0 * u.deg

print(angle.to(u.arcmin))
print(angle.to(u.arcsec))

60.0 arcmin
3600.0 arcsec


Right Ascension is often written in **hours, minutes and seconds** rather than degrees.

A full circle is 24 hours of Right Ascension, so:

$$
1\,\mathrm{hour} = 15^\circ.
$$

In [16]:
ra_angle = 5 * u.hourangle

print(ra_angle)
print(ra_angle.to(u.deg))

5.0 hourangle
74.99999999999999 deg


**Note:** due to the way Astropy stores `Quantity` values as ordinary floating-point numbers, many conversions cannot be represented exactly in
binary floating-point arithmetic, so tiny rounding artefacts may appear in the results of some calculations.

### Exercise 3

Convert:

- 30 arcsec into degrees
- 12 hours of Right Ascension into degrees.

In [17]:
# Your code here

In [18]:
# Solution
print((30. * u.arcsec).to(u.deg))
print((12. * u.hourangle).to(u.deg))

0.008333333333333333 deg
179.99999999999997 deg


# Part C — Celestial coordinates

## 7. Create a position on the sky

`SkyCoord` represents a celestial position together with its coordinate frame.

We can create a coordinate using decimal degrees:

In [19]:
coord = SkyCoord(
    ra=83.6331 * u.deg,
    dec=22.0145 * u.deg,
    frame="icrs",
)

coord

<SkyCoord (ICRS): (ra, dec) in deg
    (83.6331, 22.0145)>

We can access the Right Ascension and Declination separately.

In [20]:
print(coord.ra)
print(coord.dec)

print(coord.ra.deg)
print(coord.dec.deg)

83d37m59.16s
22d00m52.2s
83.6331
22.0145


## 8. Sexagesimal coordinates

Astronomers frequently write coordinates in forms such as:

`05h34m31.94s +22d00m52.2s`

`SkyCoord` can parse this representation directly if we tell it which units to expect.

In [21]:
coord_sexagesimal = SkyCoord(
    "05h34m31.94s +22d00m52.2s",
    frame="icrs",
)

coord_sexagesimal

<SkyCoord (ICRS): (ra, dec) in deg
    (83.63308333, 22.0145)>

In [22]:
print(coord_sexagesimal.to_string("hmsdms"))
print(coord_sexagesimal.to_string("decimal"))

05h34m31.94s +22d00m52.2s
83.6331 22.0145


### Exercise 4

Create a `SkyCoord` for:

`18h36m56.34s +38d47m01.3s`

Print its Right Ascension and Declination in decimal degrees.

In [23]:
# Your code here

In [24]:
# Solution
exercise_coord = SkyCoord("18h36m56.34s +38d47m01.3s", frame="icrs")

print(exercise_coord.ra.deg)
print(exercise_coord.dec.deg)

279.23475
38.78369444444444


## 9. Coordinate frames

The same physical direction on the sky can be represented in different coordinate systems.

Two common frames are:

- **ICRS**, normally expressed as Right Ascension and Declination
- **Galactic**, expressed as Galactic longitude `l` and latitude `b`.

Astropy handles the transformation between them.

In [25]:
galactic_coord = coord.galactic

print(galactic_coord)
print("l =", galactic_coord.l)
print("b =", galactic_coord.b)

<SkyCoord (Galactic): (l, b) in deg
    (184.55746619, -5.78434365)>
l = 184d33m26.8782929s
b = -5d47m03.63714748s


The original `coord` has not changed; we have created a different representation of the same sky position.

### Exercise 5

Transform the coordinate from Exercise 4 into Galactic coordinates and report `l` and `b` in degrees.

In [26]:
# Your code here

In [27]:
# Solution
exercise_gal = exercise_coord.galactic

print(exercise_gal.l.deg)
print(exercise_gal.b.deg)

67.44821786195831
19.23724297591511


# Part D — Separations and catalogues

## 10. Angular separation

One of the most common coordinate calculations in astronomy is the angular separation between two positions.

In [28]:
source_a = SkyCoord("05h34m31.94s +22d00m52.2s", frame="icrs")
source_b = SkyCoord("05h34m32.20s +22d00m49.0s", frame="icrs")

separation = source_a.separation(source_b)

print(separation)
print(separation.to(u.arcsec))

0d00m04.82835237s
4.82835 arcsec


The result is itself an Astropy angle/quantity, so it can be converted into whatever angular unit is useful.

## 11. Arrays of celestial coordinates

`SkyCoord` is vectorised, so a single object can represent an entire array of positions. This is especially useful for catalogues of object positions.

In [29]:
ra = np.array([83.6331, 83.6350, 83.6285, 83.6402]) * u.deg
dec = np.array([22.0145, 22.0160, 22.0095, 22.0200]) * u.deg

catalogue = SkyCoord(ra=ra, dec=dec, frame="icrs")

catalogue

<SkyCoord (ICRS): (ra, dec) in deg
    [(83.6331, 22.0145), (83.635 , 22.016 ), (83.6285, 22.0095),
     (83.6402, 22.02  )]>

We can calculate the separation between every source in the catalogue and one reference position in a single operation.

In [30]:
reference = SkyCoord(ra=83.6331 * u.deg, dec=22.0145 * u.deg)

separations = reference.separation(catalogue)

print(separations.to(u.arcsec))

[0 arcsec 8.32896 arcsec 23.6582 arcsec 30.8794 arcsec]


Boolean indexing then lets us select sources within a chosen radius.

In [31]:
nearby = separations < 30 * u.arcsec

print(nearby)
print(catalogue[nearby])

[ True  True  True False]
<SkyCoord (ICRS): (ra, dec) in deg
    [(83.6331, 22.0145), (83.635 , 22.016 ), (83.6285, 22.0095)]>


## 12. Catalogue matching

Suppose two telescopes observed approximately the same sources. Their coordinates will differ because of measurement uncertainty. `match_to_catalog_sky()` finds the nearest neighbour in catalogue B for every source in catalogue A and returns:

- `idx`: the row of the nearest source in B
- `d2d`: its on-sky angular separation
- `d3d`: a three-dimensional separation (only used when distances are included).

Nearest does **not** automatically mean plausible, so a maximum matching radius should be part of the decision-making process.

In [32]:
catalogue_a = SkyCoord(
    ra=np.array([150.0000, 150.0100, 150.0200, 150.0350]) * u.deg,
    dec=np.array([2.0000, 2.0080, 1.9950, 2.0200]) * u.deg,
)

catalogue_b = SkyCoord(
    ra=np.array([150.0204, 149.9997, 150.0355, 150.0102, 150.0800]) * u.deg,
    dec=np.array([1.9952, 2.0002, 2.0198, 2.0077, 2.0500]) * u.deg,
)

idx, d2d, _ = catalogue_a.match_to_catalog_sky(catalogue_b)

print("Nearest indices:", idx)
print("Separations:", d2d.to(u.arcsec))

Nearest indices: [1 3 0 2]
Separations: [1.29745 arcsec 1.29775 arcsec 1.60919 arcsec 1.93762 arcsec]


`idx` tells us which row in `catalogue_b` is the closest match to each source in `catalogue_a`.

A nearest neighbour is not automatically a **good** match. We normally also impose a maximum allowed separation.

In [33]:
maximum_separation = 5 * u.arcsec

good_match = d2d < maximum_separation

for i in range(len(catalogue_a)):
    if good_match[i]:
        print(
            f"A[{i}] -> B[{idx[i]}], "
            f"separation = {d2d[i].to(u.arcsec):.2f}"
        )
    else:
        print(
            f"A[{i}] has no match within {maximum_separation}"
        )

A[0] -> B[1], separation = 1.30 arcsec
A[1] -> B[3], separation = 1.30 arcsec
A[2] -> B[0], separation = 1.61 arcsec
A[3] -> B[2], separation = 1.94 arcsec


### Exercise 6

Change the maximum matching radius to:

- 1 arcsec
- 3 arcsec
- 10 arcsec.

How does the number of accepted matches change? What aspects of the observations would this maximum radius depend on?

In [34]:
# Your code here

# Final Task: Catalogue Matching

Two small catalogues are given below. Catalogue 1 might represent detections from an optical-wavelength image and Catalogue 2 detections from a radio-wavelength image.

Your task is to:

1. create a `SkyCoord` object for each catalogue
2. convert Catalogue 1 into Galactic coordinates
3. find the nearest Catalogue 2 source to every Catalogue 1 source
4. report the nearest-neighbour separations in arcsec
5. accept matches only if the separation is less than 2 arcsec
6. identify which Catalogue 1 sources do **not** have an acceptable counterpart.

Think about what the matching radius means physically.

In [35]:
source_id_1 = np.array(["O1", "O2", "O3", "O4", "O5"])
ra_1 = np.array([210.80230, 210.81100, 210.82400, 210.84000, 210.85700])
dec_1 = np.array([54.34890, 54.35200, 54.34450, 54.36000, 54.37100])

source_id_2 = np.array(["R1", "R2", "R3", "R4", "R5", "R6"])
ra_2 = np.array([210.82415, 210.80220, 210.85680, 210.81125, 210.90000, 210.83910])
dec_2 = np.array([54.34460, 54.34875, 54.37110, 54.35210, 54.40000, 54.35880])

In [36]:
# Final exercise workspace

# catalogue_1 = ...
# catalogue_2 = ...
# ...

In [37]:
# Solution 

catalogue_1 = SkyCoord(ra=ra_1 * u.deg, dec=dec_1 * u.deg)
catalogue_2 = SkyCoord(ra=ra_2 * u.deg, dec=dec_2 * u.deg)

print("Catalogue 1 in Galactic coordinates:")
for name, c in zip(source_id_1, catalogue_1.galactic):
    print(name, f"l={c.l.deg:.4f} deg", f"b={c.b.deg:.4f} deg")

idx, d2d, _ = catalogue_1.match_to_catalog_sky(catalogue_2)
accepted = d2d < 2 * u.arcsec

print("\nMatches:")
for i, name in enumerate(source_id_1):
    if accepted[i]:
        print(
            f"{name} -> {source_id_2[idx[i]]}: "
            f"{d2d[i].to_value(u.arcsec):.2f} arcsec"
        )
    else:
        print(
            f"{name}: no acceptable counterpart "
            f"(nearest = {d2d[i].to_value(u.arcsec):.2f} arcsec)"
        )

Catalogue 1 in Galactic coordinates:
O1 l=102.0373 deg b=59.7713 deg
O2 l=102.0322 deg b=59.7660 deg
O3 l=102.0115 deg b=59.7681 deg
O4 l=102.0127 deg b=59.7500 deg
O5 l=102.0081 deg b=59.7354 deg

Matches:
O1 -> R2: 0.58 arcsec
O2 -> R4: 0.64 arcsec
O3 -> R1: 0.48 arcsec
O4: no acceptable counterpart (nearest = 4.71 arcsec)
O5 -> R3: 0.55 arcsec


# Assessment

You should be prepared to discuss your code for the final task during the viva for this lab. Bring a copy of this notebook with your completed analysis included.